# Customer Churn Prediction (IBM Telco dataset)
Goal: predict whether a customer will leave (Churn = Yes/No). This is SUPERVISED learning:
we have input features (X) and a known label (y) to learn from.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")  # lets the script save figures without opening windows
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, RocCurveDisplay,
                             ConfusionMatrixDisplay, r2_score)

RANDOM_STATE = 42  # fixed seed so results are the same every run

## Step 1: Load and understand the data

In [2]:
df = pd.read_csv("data/telco.csv")
print("Shape:", df.shape)
print(df.dtypes, "\n")

# TotalCharges is stored as text; blank strings (new customers with tenure 0) become NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing values:\n", df.isnull().sum()[df.isnull().sum() > 0], "\n")
print("Duplicate customerIDs:", df["customerID"].duplicated().sum())

# Target: Yes -> 1, No -> 0
df["Churn"] = (df["Churn"] == "Yes").astype(int)
print("\nClass balance (proportion):\n", df["Churn"].value_counts(normalize=True).round(3))

# customerID is just a label, not a real feature, so we drop it
df = df.drop(columns=["customerID"])

# Features (X) and target (y)
X = df.drop(columns=["Churn"])
y = df["Churn"]

numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
categorical_cols = [c for c in X.columns if c not in numeric_cols]
print("\nNumeric features:", numeric_cols)
print("Categorical features:", categorical_cols)

# Two simple charts
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
df["Churn"].value_counts().rename({0: "No", 1: "Yes"}).plot.bar(ax=ax[0], color=["#4c72b0", "#dd8452"])
ax[0].set_title("Class balance (Churn)")
df.groupby("Contract")["Churn"].mean().plot.bar(ax=ax[1], color="#55a868")
ax[1].set_title("Churn rate by contract type")
ax[1].set_ylabel("Churn rate")
plt.tight_layout()
plt.savefig("reports/figures/01_eda.png", dpi=120)
plt.close()

Shape: (7043, 21)
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object 

Missing values:
 TotalCharges    11
dtype: int64 

Duplicate customerIDs: 0

Class balance (proportion):
 Churn
0    0.735
1    0.265
Name: proportion, dtype: float64

Numeric features: ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'O

## Step 2: Split the data (stratified 70 / 15 / 15)
Split FIRST, before any preprocessing, so no information from validation/test leaks into training.
stratify=y keeps the same churn percentage in every split.

In [3]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE)

for name, part in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(f"{name:10s} rows={len(part):5d}  churn rate={part.mean():.3f}")

Train      rows= 4930  churn rate=0.265
Validation rows= 1056  churn rate=0.265
Test       rows= 1057  churn rate=0.266


## Step 3: Preprocessing + baseline (Logistic Regression)
The preprocessor is part of the Pipeline, so it is fitted ONLY on the training data
(and re-fitted inside each cross-validation fold). This prevents leakage.

In [4]:
def make_preprocessor():
    numeric_steps = Pipeline([
        ("impute", SimpleImputer(strategy="median")),   # fill missing numbers with the median
        ("scale", StandardScaler()),                    # mean 0, std 1
    ])
    categorical_steps = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),  # text categories -> 0/1 columns
    ])
    return ColumnTransformer([
        ("num", numeric_steps, numeric_cols),
        ("cat", categorical_steps, categorical_cols),
    ])


def make_pipeline(model):
    return Pipeline([("prep", make_preprocessor()), ("model", model)])


def get_scores(pipe, X_part, y_part, threshold=0.5):
    """Return precision, recall, F1 and ROC-AUC for a fitted pipeline."""
    proba = pipe.predict_proba(X_part)[:, 1]          # probability of churn
    pred = (proba >= threshold).astype(int)
    return {
        "precision": precision_score(y_part, pred),
        "recall": recall_score(y_part, pred),
        "f1": f1_score(y_part, pred),
        "roc_auc": roc_auc_score(y_part, proba),
    }


# class_weight="balanced" tells the model that churners are rarer, so mistakes on them cost more
baseline = make_pipeline(LogisticRegression(max_iter=1000, class_weight="balanced"))
baseline.fit(X_train, y_train)
print("Baseline (Logistic Regression) on validation:")
print({k: round(v, 3) for k, v in get_scores(baseline, X_val, y_val).items()})

Baseline (Logistic Regression) on validation:
{'precision': 0.506, 'recall': 0.814, 'f1': 0.624, 'roc_auc': 0.845}


## Step 4: Compare three models (same split, same preprocessing, same metrics)
We also compare TRAIN vs VALIDATION F1. A big gap means overfitting.

In [5]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
}

rows = []
for name, model in models.items():
    pipe = make_pipeline(model)
    pipe.fit(X_train, y_train)
    tr = get_scores(pipe, X_train, y_train)
    va = get_scores(pipe, X_val, y_val)
    rows.append({
        "model": name,
        "train_f1": tr["f1"], "val_f1": va["f1"], "f1_gap": tr["f1"] - va["f1"],
        "val_precision": va["precision"], "val_recall": va["recall"], "val_roc_auc": va["roc_auc"],
    })

comparison = pd.DataFrame(rows).round(3)
comparison.to_csv("reports/model_comparison.csv", index=False)
print(comparison.to_string(index=False))

              model  train_f1  val_f1  f1_gap  val_precision  val_recall  val_roc_auc
Logistic Regression     0.628   0.624   0.004          0.506       0.814        0.845
      Decision Tree     0.996   0.474   0.522          0.460       0.489        0.641
      Random Forest     0.983   0.599   0.384          0.547       0.661        0.816


## Step 5a: Overfitting and regularization curves
- Logistic Regression: smaller C = STRONGER regularization (simpler model)
- Decision Tree: larger max_depth = MORE complex model

In [6]:
c_values = [0.001, 0.01, 0.1, 1, 10, 100]
lr_train, lr_val = [], []
for c in c_values:
    p = make_pipeline(LogisticRegression(C=c, max_iter=1000, class_weight="balanced"))
    p.fit(X_train, y_train)
    lr_train.append(get_scores(p, X_train, y_train)["f1"])
    lr_val.append(get_scores(p, X_val, y_val)["f1"])

depths = [1, 2, 3, 4, 5, 6, 8, 10, 15, 20]
dt_train, dt_val = [], []
for d in depths:
    p = make_pipeline(DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE, class_weight="balanced"))
    p.fit(X_train, y_train)
    dt_train.append(get_scores(p, X_train, y_train)["f1"])
    dt_val.append(get_scores(p, X_val, y_val)["f1"])

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(c_values, lr_train, "o-", label="train")
ax[0].plot(c_values, lr_val, "o-", label="validation")
ax[0].set_xscale("log"); ax[0].set_xlabel("C (smaller = more regularization)")
ax[0].set_ylabel("F1"); ax[0].set_title("Logistic Regression"); ax[0].legend()
ax[1].plot(depths, dt_train, "o-", label="train")
ax[1].plot(depths, dt_val, "o-", label="validation")
ax[1].set_xlabel("max_depth (bigger = more complex)")
ax[1].set_ylabel("F1"); ax[1].set_title("Decision Tree"); ax[1].legend()
plt.tight_layout()
plt.savefig("reports/figures/02_overfitting_curves.png", dpi=120)
plt.close()
print("Best tree depth on validation:", depths[int(np.argmax(dt_val))])

Best tree depth on validation: 3


## Step 5b: Tune Random Forest with stratified cross-validation (on TRAINING data only)
Metric chosen in advance: F1. Small grid so it runs quickly and is easy to explain.

In [7]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [5, 10, None],
    "model__min_samples_leaf": [1, 5, 10],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rf_pipe = make_pipeline(RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced"))

search = GridSearchCV(rf_pipe, param_grid, scoring="f1", cv=cv, n_jobs=-1)
search.fit(X_train, y_train)
print("Best parameters:", search.best_params_)
print("Best CV F1:", round(search.best_score_, 3))

# Is the improvement consistent across folds? Compare default vs tuned fold by fold.
default_cv = cross_val_score(rf_pipe, X_train, y_train, scoring="f1", cv=cv)
tuned_cv = cross_val_score(search.best_estimator_, X_train, y_train, scoring="f1", cv=cv)
print("Default RF F1 per fold:", default_cv.round(3), "mean", default_cv.mean().round(3))
print("Tuned   RF F1 per fold:", tuned_cv.round(3), "mean", tuned_cv.mean().round(3))

best_rf = search.best_estimator_
tuned_val = get_scores(best_rf, X_val, y_val)
print("Tuned RF on validation:", {k: round(v, 3) for k, v in tuned_val.items()})

Best parameters: {'model__max_depth': None, 'model__min_samples_leaf': 10, 'model__n_estimators': 100}
Best CV F1: 0.633
Default RF F1 per fold: [0.594 0.625 0.596 0.595 0.63 ] mean 0.608
Tuned   RF F1 per fold: [0.61  0.657 0.614 0.623 0.66 ] mean 0.633
Tuned RF on validation: {'precision': 0.524, 'recall': 0.821, 'f1': 0.64, 'roc_auc': 0.844}


## Step 6: Choose the final model and threshold using VALIDATION data only
Candidates: baseline LR, default tree, default RF, tuned RF. Pick the best validation F1.
Then pick the probability threshold (instead of blindly using 0.5) that maximizes validation F1.

In [8]:
candidates = {
    "Logistic Regression": baseline,
    "Tuned Random Forest": best_rf,
}
final_name = max(candidates, key=lambda n: get_scores(candidates[n], X_val, y_val)["f1"])
final_model = candidates[final_name]
print("Selected model:", final_name)

val_proba = final_model.predict_proba(X_val)[:, 1]
thresholds = np.arange(0.2, 0.81, 0.05)
f1_by_threshold = [f1_score(y_val, (val_proba >= t).astype(int)) for t in thresholds]
best_threshold = float(thresholds[int(np.argmax(f1_by_threshold))])
print("Chosen threshold:", round(best_threshold, 2))

Selected model: Tuned Random Forest
Chosen threshold: 0.55


## Step 7: Final evaluation on the untouched TEST set (used only once)

In [9]:
test_scores = get_scores(final_model, X_test, y_test, threshold=best_threshold)
print("TEST scores:", {k: round(v, 3) for k, v in test_scores.items()})

test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)
print("\nConfusion matrix:\n", confusion_matrix(y_test, test_pred))
print("\n", classification_report(y_test, test_pred, target_names=["No churn", "Churn"]))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, display_labels=["No", "Yes"], ax=ax[0], colorbar=False)
ax[0].set_title("Confusion matrix (test)")
RocCurveDisplay.from_predictions(y_test, test_proba, ax=ax[1], name=final_name)
ax[1].plot([0, 1], [0, 1], "k--", label="random")
ax[1].set_title("ROC curve (test)"); ax[1].legend()
plt.tight_layout()
plt.savefig("reports/figures/03_final_evaluation.png", dpi=120)
plt.close()

TEST scores: {'precision': 0.564, 'recall': 0.701, 'f1': 0.625, 'roc_auc': 0.843}

Confusion matrix:
 [[624 152]
 [ 84 197]]

               precision    recall  f1-score   support

    No churn       0.88      0.80      0.84       776
       Churn       0.56      0.70      0.63       281

    accuracy                           0.78      1057
   macro avg       0.72      0.75      0.73      1057
weighted avg       0.80      0.78      0.78      1057



## Step 8: What drives churn? (interpretability)

In [10]:
feature_names = baseline.named_steps["prep"].get_feature_names_out()

# Logistic regression: positive coefficient = more likely to churn
coefs = pd.Series(baseline.named_steps["model"].coef_[0], index=feature_names).sort_values()
top_lr = pd.concat([coefs.head(8), coefs.tail(8)])

# Random forest: importance = how much a feature helps the trees split
rf_imp = pd.Series(best_rf.named_steps["model"].feature_importances_, index=feature_names)
top_rf = rf_imp.sort_values().tail(12)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
top_lr.plot.barh(ax=ax[0], color=["#4c72b0" if v < 0 else "#dd8452" for v in top_lr])
ax[0].set_title("Logistic Regression coefficients\n(orange = increases churn, blue = decreases)")
top_rf.plot.barh(ax=ax[1], color="#55a868")
ax[1].set_title("Random Forest feature importance")
plt.tight_layout()
plt.savefig("reports/figures/04_feature_effects.png", dpi=120)
plt.close()
print("Top features (Random Forest):\n", top_rf.sort_values(ascending=False).head(5).round(3))

Top features (Random Forest):
 cat__Contract_Month-to-month    0.133
num__tenure                     0.125
num__TotalCharges               0.090
cat__OnlineSecurity_No          0.080
cat__Contract_Two year          0.074
dtype: float64
